# Swiss Legal Retrieval — Kaggle Submission Notebook

Offline notebook. No internet access at runtime.

**Pre-requisites (upload as Kaggle datasets before submitting):**
- Fine-tuned model directory → dataset name `qwen-swiss-legal`
- (Optional) Pre-built FAISS index → dataset name `swiss-legal-index`
  - Skip corpus embedding step at runtime if index is available


In [ ]:
!pip install sentence-transformers faiss-cpu datasets -q

In [ ]:
import sys
sys.path.append('/kaggle/input/swiss-legal-src/src')

import pandas as pd
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
import faiss

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
DATA_DIR   = Path('/kaggle/input/llm-agentic-legal-information-retrieval')
MODEL_DIR  = Path('/kaggle/input/qwen-swiss-legal')       # fine-tuned model
INDEX_DIR  = Path('/kaggle/input/swiss-legal-index')      # optional pre-built index
OUTPUT_DIR = Path('/kaggle/working')

# Fall back to base model if fine-tuned not available
MODEL_PATH = MODEL_DIR if MODEL_DIR.exists() else 'Qwen/Qwen3-Embedding-0.6B'

# Qwen3-Embedding: prepend instruction to queries only (not documents)
QUERY_INSTRUCTION = (
    'Instruct: Given an English legal question, retrieve the most relevant '
    'Swiss legal sources (statutes and court decisions).\nQuery: '
)

TOP_K           = 30    # candidates to retrieve; adaptive threshold cuts this down
GAP_FRACTION    = 0.15  # score-gap fraction for adaptive cutoff
ENCODE_BATCH    = 128

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────
test_df  = pd.read_csv(DATA_DIR / 'test.csv')
laws_df  = pd.read_csv(DATA_DIR / 'laws_de.csv')
court_df = pd.read_csv(DATA_DIR / 'court_considerations.csv')

corpus_df = pd.concat([laws_df, court_df], ignore_index=True)
corpus_df = corpus_df.dropna(subset=['citation', 'text']).drop_duplicates('citation').reset_index(drop=True)

print(f'Test queries : {len(test_df)}')
print(f'Corpus size  : {len(corpus_df)}')

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────
print(f'Loading model from: {MODEL_PATH}')
model = SentenceTransformer(str(MODEL_PATH))

In [ ]:
# ── Build or load FAISS index ─────────────────────────────────────────────
CITATIONS = corpus_df['citation'].tolist()

if (INDEX_DIR / 'corpus.index').exists():
    print('Loading pre-built FAISS index ...')
    index = faiss.read_index(str(INDEX_DIR / 'corpus.index'))
    print(f'Index loaded: {index.ntotal} vectors')
else:
    print('Encoding corpus (no pre-built index found) ...')
    corpus_embs = model.encode(
        corpus_df['text'].tolist(),
        batch_size=ENCODE_BATCH,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    dim = corpus_embs.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(corpus_embs)
    print(f'Index built: {index.ntotal} vectors')

In [ ]:
# ── Encode queries ────────────────────────────────────────────────────────
queries = test_df['query'].tolist()
print(f'Encoding {len(queries)} queries ...')
query_embs = model.encode(
    queries,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    prompt=QUERY_INSTRUCTION,
).astype(np.float32)

In [ ]:
# ── Retrieve with adaptive threshold ──────────────────────────────────────
scores_all, indices_all = index.search(query_embs, TOP_K)


def adaptive_cutoff(citations, scores, gap_fraction):
    """Return citations up to the largest score gap above gap_fraction * top_score."""
    if len(citations) <= 1:
        return citations
    threshold = gap_fraction * float(scores[0])
    best_cut, best_gap = len(citations), 0.0
    for i in range(len(citations) - 1):
        gap = float(scores[i]) - float(scores[i + 1])
        if gap > threshold and gap > best_gap:
            best_gap, best_cut = gap, i + 1
    return citations[:best_cut]


rows = []
for qid, idx_row, score_row in zip(test_df['query_id'], indices_all, scores_all):
    candidates = [CITATIONS[i] for i in idx_row if i >= 0]
    predicted  = adaptive_cutoff(candidates, score_row, GAP_FRACTION)
    rows.append({'query_id': qid, 'predicted_citations': ';'.join(predicted)})

submission = pd.DataFrame(rows)
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
print('submission.csv saved!')
submission.head()